# Train and publish the release classifier (model soup)

This trains the classifier that the `citeaudit` package downloads and publishes it on the Hugging
Face Hub. It is the NLI model of `results/nli_classifier_v1.md`, with the same data and settings.

Averaging 3 seeds' predictions beats a single seed on test (0.680 / 0.575 against 0.668 / 0.556),
but three models would triple the download and the run time. So this trains 3 seeds and averages
their **weights** into one model (`scripts/soup.py`, a "model soup"). **The rule is fixed in advance
and uses dev only:** the soup is published if its dev macro-F1 is at least the mean of the three
seeds'; otherwise the seed with the best dev score is published. Test scores are reported for every
candidate but decide nothing.

About 2 hours, and about 4 GB of Google Drive space for the models.

**Before running:**
1. On huggingface.co, signed in as the account the model should belong to, open Settings →
   Access Tokens → Create new token, of type **Write**.
2. In Colab, click the **key icon** in the left sidebar (Secrets) → **Add new secret**. Name it
   `HF_TOKEN`, paste the token, and switch on **Notebook access**.
3. Runtime → Change runtime type → **T4 GPU**, then Runtime → Run all, and allow Google Drive access.

The model goes to `<your username>/citeaudit-nli` and a copy stays in `MyDrive/CiteAudit_runs/release`.
Without the token, nothing is published.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
from google.colab import drive
drive.mount('/content/drive')
RES = '/content/drive/MyDrive/CiteAudit_runs/release'
import os; os.makedirs(RES, exist_ok=True)

%cd /content
!rm -rf CiteAudit && git clone -q https://github.com/SapnilPatel/CiteAudit.git
%cd /content/CiteAudit
!git log -1 --oneline
!pip -q install "transformers>=4.44" "tokenizers>=0.19" sentencepiece protobuf huggingface_hub

import subprocess, json, statistics
def run(cmd):
    noise = ('UNEXPECTED', 'MISSING', 'LOAD REPORT', 'Key ', '----', 'Notes', '- UNEXP', '- MISS',
             'Loading weights', 'Warning', 'warn(')
    p = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in p.stdout:
        if not any(n in line for n in noise):
            print(line, end='')
    if p.wait():
        raise RuntimeError(f'failed ({p.returncode}): {cmd}')

T = 'data_train/target'
def train(name, seed, extra):
    out = f'{RES}/{name}'
    if os.path.exists(f'{out}/config.json'):
        print(f'{name}: already done'); return
    run(f'python scripts/train_nli.py --train {T}/ours_20r_train_cv --dev {T}/ours_20r '
        f'--epochs 3 --batch 2 --accum 4 --seed {seed} --amp '
        f'--out checkpoints/{name} --pred-dir {out}.part {extra}')
    run(f'rm -rf checkpoints/{name}')
    os.rename(f'{out}.part', out)

LABELS = ['ACCURATE', 'NOT_ACCURATE', 'NOT ENOUGH INFO']
def scores(rows):
    f1 = []
    for c in LABELS:
        tp = sum(r['gold'] == c and r['prediction'] == c for r in rows)
        fp = sum(r['gold'] != c and r['prediction'] == c for r in rows)
        fn = sum(r['gold'] == c and r['prediction'] != c for r in rows)
        f1.append(2 * tp / (2 * tp + fp + fn) if tp else 0.)
    return [sum(r['gold'] == r['prediction'] for r in rows) / len(rows), sum(f1) / 3] + f1
read = lambda p: [json.loads(l) for l in open(p)]
COLS = ['accuracy', 'macro-F1', 'ACC F1', 'N_ACC F1', 'NEI F1']

## Train 3 seeds, saving each model

In [ ]:
BASE_ARGS = (f'--train {T}/ours_20r_train_cv --dev {T}/ours_20r --epochs 3 --batch 2 --accum 4 --amp '
             f'--predict test={T}/ours_20r_test')
for seed in (0, 1, 2):
    m = f'{RES}/m_s{seed}'
    if os.path.exists(f'{m}/model.safetensors'):
        print(f'seed {seed}: already done'); continue
    run(f'python scripts/train_nli.py {BASE_ARGS} --seed {seed} --out checkpoints/s{seed} '
        f'--pred-dir {RES}/preds_s{seed} --save-model {m}.part')
    run(f'rm -rf checkpoints/s{seed}')
    os.rename(f'{m}.part', m)

## Average the weights, score the soup, apply the rule

In [ ]:
SOUP = f'{RES}/soup'
if not os.path.exists(f'{SOUP}/model.safetensors'):
    run(f'python scripts/soup.py --models {RES}/m_s0 {RES}/m_s1 {RES}/m_s2 --out {SOUP}')
if not os.path.exists(f'{RES}/preds_soup/config.json'):
    # --epochs 0 only scores the loaded model: dev, then test
    run(f'python scripts/train_nli.py --model {SOUP} {BASE_ARGS} --epochs 0 --out checkpoints/soup '
        f'--pred-dir {RES}/preds_soup')
    run('rm -rf checkpoints/soup')

dev = {f's{s}': json.load(open(f'{RES}/preds_s{s}/config.json'))['best_dev_macro_f1'] for s in (0, 1, 2)}
dev['soup'] = json.load(open(f'{RES}/preds_soup/config.json'))['best_dev_macro_f1']
test = {k: scores(read(f'{RES}/preds_{k}/ours_20r_test.jsonl')) for k in dev}
members = [read(f'{RES}/preds_s{s}/ours_20r_test.jsonl') for s in (0, 1, 2)]
ens = [{'gold': rs[0]['gold'], 'prediction': max(LABELS, key=lambda c: sum(r['probs'][c] for r in rs))}
       for rs in zip(*members)]
print(f'{"model":28s} {"dev macro-F1":>13s}   test ' + ' '.join(f'{c:>9s}' for c in COLS))
for k in dev:
    print(f'{k:28s} {dev[k]:13.3f}        ' + ' '.join(f'{x:9.3f}' for x in test[k]))
print(f'{"3 seeds, predictions averaged":28s} {"":13s}        ' + ' '.join(f'{x:9.3f}' for x in scores(ens)))

mean_dev = statistics.mean(dev[f's{s}'] for s in (0, 1, 2))
if dev['soup'] >= mean_dev:
    CHOICE, NOTE = 'soup', ('This checkpoint is a uniform average of the weights of 3 fine-tuning runs '
                            '(seeds 0, 1, 2), a "model soup", which gives most of an ensemble\'s accuracy at one model\'s cost.')
else:
    best = max((0, 1, 2), key=lambda s: dev[f's{s}'])
    CHOICE, NOTE = f's{best}', f'This checkpoint is the fine-tuning run with seed {best}, the best of 3 on dev.'
print(f'\nsoup dev {dev["soup"]:.3f} against mean of seeds {mean_dev:.3f}: publishing {CHOICE}')
MODEL_DIR = SOUP if CHOICE == 'soup' else f'{RES}/m_{CHOICE}'
json.dump({'choice': CHOICE, 'dev': dev, 'test': test, 'ensemble_test': scores(ens)},
          open(f'{RES}/release_summary.json', 'w'), indent=2)
json.dump(dict(zip(COLS, test[CHOICE])), open(f'{MODEL_DIR}/test_metrics.json', 'w'), indent=2)

## Publish to the Hugging Face Hub

In [ ]:
from huggingface_hub import HfApi
try:
    from google.colab import userdata
    token = userdata.get('HF_TOKEN')
except Exception:
    token = None
if not token:
    print('No HF_TOKEN secret found: nothing published. The model is on Drive at', MODEL_DIR)
else:
    api = HfApi(token=token)
    repo = f"{api.whoami()['name']}/citeaudit-nli"
    card = open('release/nli_model_card.md').read().replace('REPO_ID', repo).replace('CHECKPOINT_NOTE', NOTE)
    open(f'{MODEL_DIR}/README.md', 'w').write(card)
    api.create_repo(repo, exist_ok=True)
    api.upload_folder(folder_path=MODEL_DIR, repo_id=repo, commit_message='CiteAudit release classifier',
                      allow_patterns=['*.json', '*.safetensors', 'README.md'])
    print('published: https://huggingface.co/' + repo)

## Download

Zips the predictions and logs (not the models) to `release.zip`. Put it in the repository root and
tell Claude the Hub link printed above.

In [ ]:
!cd /content/drive/MyDrive/CiteAudit_runs && zip -qr /content/release.zip release -x '*.safetensors' -x '*.part/*'
from google.colab import files
files.download('/content/release.zip')